In [1]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    confusion_matrix,
    classification_report,
)

model_data = pd.read_csv("../data/processed/advanced_match_features.csv")

model_data["date"] = pd.to_datetime(model_data["date"])

model_data.head()

,date,home_team,away_team,tournament,result,elo_difference,home_advantage,recent_win_rate_difference,recent_goals_for_difference,recent_goals_against_difference,recent_goal_difference_difference,rest_days_difference,streak_difference,is_world_cup,is_qualification,is_friendly,tournament_importance,head_to_head_difference,attack_rating_difference,defense_rating_difference
0,1872-11-30,Scotland,England,Friendly,draw,0.000000,1,0.0,0.000000,0.000000,0.000000,0,0,0,0,1,1,0,0.000000,0.000000
1,1873-03-08,England,Scotland,Friendly,home_win,0.000000,1,0.0,0.000000,0.000000,0.000000,0,0,0,0,1,1,0,0.000000,0.000000
2,1874-03-07,Scotland,England,Friendly,home_win,-10.000000,1,-0.5,-1.000000,1.000000,-2.000000,0,-2,0,0,1,1,-1,-0.100000,-0.100000
3,1875-03-06,England,Scotland,Friendly,draw,-0.287744,1,0.0,0.333333,-0.333333,0.666667,0,-2,0,0,1,1,0,0.038776,0.038776
4,1876-03-04,Scotland,England,Friendly,home_win,0.279462,1,0.0,-0.250000,0.250000,-0.500000,0,0,0,0,1,1,0,-0.034255,-0.034255


In [2]:
features = [
    "elo_difference",
    "home_advantage",
    "recent_win_rate_difference",
    "recent_goals_for_difference",
    "recent_goals_against_difference",
    "recent_goal_difference_difference",
    "rest_days_difference",
    "streak_difference",
    "is_world_cup",
    "is_qualification",
    "is_friendly",
    "tournament_importance",
    "head_to_head_difference",
    "attack_rating_difference",
    "defense_rating_difference",
]

train = model_data[model_data["date"] < "2018-01-01"]
test = model_data[model_data["date"] >= "2018-01-01"]

X_train = train[features]
y_train = train["result"]

X_test = test[features]
y_test = test["result"]

print("Training rows:", len(train))
print("Testing rows:", len(test))
print("Features:", len(features))

Training rows: 41297
Testing rows: 8199
Features: 15


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=5000,
        random_state=42,
        class_weight="balanced",
    ))
])

logistic_model.fit(X_train, y_train)

logistic_predictions = logistic_model.predict(X_test)
logistic_probabilities = logistic_model.predict_proba(X_test)

In [8]:
logistic_accuracy = accuracy_score(
    y_test,
    logistic_predictions
)

logistic_loss = log_loss(
    y_test,
    logistic_probabilities,
    labels=logistic_model.classes_
)

print("Logistic Regression Accuracy:", logistic_accuracy)
print("Logistic Regression Log Loss:", logistic_loss)
print(classification_report(
    y_test,
    logistic_predictions
))
confusion_matrix(
    y_test,
    logistic_predictions,
    labels=logistic_model.classes_
)

Logistic Regression Accuracy: 0.5811684351750214
Logistic Regression Log Loss: 0.8998393577184494
              precision    recall  f1-score   support

    away_win       0.56      0.67      0.61      2386
        draw       0.29      0.20      0.24      1903
    home_win       0.70      0.71      0.70      3910

    accuracy                           0.58      8199
   macro avg       0.51      0.53      0.52      8199
weighted avg       0.56      0.58      0.57      8199



array([[1588,  385,  413],
       [ 706,  390,  807],
       [ 551,  572, 2787]])

In [10]:
random_forest_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    class_weight="balanced",
)

random_forest_model.fit(X_train, y_train)

random_forest_predictions = random_forest_model.predict(X_test)

random_forest_probabilities = random_forest_model.predict_proba(X_test)
print(random_forest_predictions[:5])

print()

print(random_forest_probabilities[:5])

['draw' 'draw' 'away_win' 'away_win' 'draw']

[[0.32410605 0.34518678 0.33070717]
 [0.31850995 0.38643201 0.29505804]
 [0.47327273 0.33417091 0.19255637]
 [0.5992814  0.30283386 0.09788475]
 [0.32500714 0.41833399 0.25665887]]


In [13]:
random_forest_accuracy = accuracy_score(
    y_test,
    random_forest_predictions
)

random_forest_loss = log_loss(
    y_test,
    random_forest_probabilities,
    labels=random_forest_model.classes_
)

print("Random Forest Accuracy:", random_forest_accuracy)
print("Random Forest Log Loss:", random_forest_loss)
print(classification_report(
    y_test,
    random_forest_predictions
))
confusion_matrix(
    y_test,
    random_forest_predictions,
    labels=random_forest_model.classes_
)

Random Forest Accuracy: 0.5805586047078912
Random Forest Log Loss: 0.8970906378035062
              precision    recall  f1-score   support

    away_win       0.57      0.64      0.60      2386
        draw       0.30      0.25      0.28      1903
    home_win       0.70      0.70      0.70      3910

    accuracy                           0.58      8199
   macro avg       0.52      0.53      0.53      8199
weighted avg       0.57      0.58      0.57      8199



array([[1532,  450,  404],
       [ 655,  485,  763],
       [ 496,  671, 2743]])

In [14]:
gradient_boosting_model = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42,
)

gradient_boosting_model.fit(X_train, y_train)

gradient_boosting_predictions = gradient_boosting_model.predict(X_test)

gradient_boosting_probabilities = gradient_boosting_model.predict_proba(X_test)

In [15]:
gradient_boosting_accuracy = accuracy_score(
    y_test,
    gradient_boosting_predictions
)

gradient_boosting_loss = log_loss(
    y_test,
    gradient_boosting_probabilities,
    labels=gradient_boosting_model.classes_
)

print("Gradient Boosting Accuracy:", gradient_boosting_accuracy)
print("Gradient Boosting Log Loss:", gradient_boosting_loss)

Gradient Boosting Accuracy: 0.6036101963654105
Gradient Boosting Log Loss: 0.8720783814833104


In [16]:
print(classification_report(
    y_test,
    gradient_boosting_predictions,
    zero_division=0
))
confusion_matrix(
    y_test,
    gradient_boosting_predictions,
    labels=gradient_boosting_model.classes_
)


              precision    recall  f1-score   support

    away_win       0.59      0.61      0.60      2386
        draw       0.18      0.00      0.00      1903
    home_win       0.61      0.89      0.73      3910

    accuracy                           0.60      8199
   macro avg       0.46      0.50      0.44      8199
weighted avg       0.50      0.60      0.52      8199



array([[1466,    7,  913],
       [ 605,    3, 1295],
       [ 423,    7, 3480]])

In [17]:
model_results = [
    {
        "model": "Logistic Regression",
        "accuracy": logistic_accuracy,
        "log_loss": logistic_loss,
    },
    {
        "model": "Random Forest",
        "accuracy": random_forest_accuracy,
        "log_loss": random_forest_loss,
    },
    {
        "model": "Gradient Boosting",
        "accuracy": gradient_boosting_accuracy,
        "log_loss": gradient_boosting_loss,
    },
]

results_table = pd.DataFrame(model_results)

results_table.sort_values("log_loss")

,model,accuracy,log_loss
2,Gradient Boosting,0.603610,0.872078
1,Random Forest,0.580559,0.897091
0,Logistic Regression,0.581168,0.899839


In [18]:
import joblib

best_model = gradient_boosting_model

joblib.dump(
    best_model,
    "../models/best_match_prediction_model.pkl"
)

joblib.dump(
    features,
    "../models/model_features.pkl"
)

print("Best model saved.")

print("Feature list saved.")

Best model saved.
Feature list saved.
